# Mozar modeling

Three predictors on the same posts and the same splits:
- **M0**, image only: FLAME and TruFor scores.
- **M1**, history only: the poster's past, as features from `graph.py features`.
- **M2**, both combined.

Posts by one account, and posts of one picture, always stay on the same side of a split.
Run on the sample with `MOZAR_DATA=sample_data jupyter lab`; on the cluster, the default is `data/`.

In [ ]:
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
from config import DATA, RESULTS
RESULTS.mkdir(exist_ok=True)
SEED = 0

feat = pd.read_csv(DATA / "features.csv", dtype={"pair_id": str})
feat["y"] = (feat.label == "manipulated").astype(int)
feat.label.value_counts()

## 1. Data

In [ ]:
print(pd.crosstab(feat.subreddit, feat.label))
feat.groupby("label")[["n_posts", "n_comments", "n_subreddits", "prior_repost_share", "has_history"]].mean().round(2)

In [ ]:
from IPython.display import Image, display
dec = pd.read_csv(DATA / "labels/decisions.csv") if (DATA / "labels/decisions.csv").exists() else pd.DataFrame()
for pid in dec[dec.decision == "labeled"].pair_id.head(3) if len(dec) else []:
    display(Image(str(DATA / f"pairs/outline/{pid}_a.jpg"), width=350), Image(str(DATA / f"pairs/outline/{pid}_b.jpg"), width=350))

## 2. Working subset

A random, shuffled sample for quick runs. Every manipulated post is kept unless `FRACTION` is below 1.
`NEG_PER_POS` caps the authentic posts per manipulated post (`None` keeps them all).
Manipulated posts stay rare either way, so every model below weights the classes:
an error on a manipulated post costs as much in total as all the errors on authentic posts.

In [ ]:
FRACTION, NEG_PER_POS = 1.0, None

pos, neg = feat[feat.y == 1], feat[feat.y == 0]
if NEG_PER_POS:
    neg = neg.sample(min(len(neg), NEG_PER_POS * len(pos)), random_state=SEED)
data = pd.concat([pos.sample(frac=FRACTION, random_state=SEED), neg.sample(frac=FRACTION, random_state=SEED)])
data = data.sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle
print(f"{len(data)} posts, {data.y.sum()} manipulated ({data.y.mean():.1%})")

## 3. Splits and scoring

Groups join every post that shares an account or a picture, so neither can sit on both sides of a split.
Scores are AUROC and average precision, the second because manipulated posts are rare.

In [ ]:
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
for a, p, img in zip(data.author, data.pair_id.fillna(""), data.img):
    parent[find("a:" + a)] = find("i:" + (p or img))
data["group"] = [find("a:" + a) for a in data.author]

HIST = ["n_posts", "n_comments", "n_subreddits", "subreddit_entropy", "share_image_posts",
        "days_observed", "posts_per_day", "n_prior_pool_posts", "prior_repost_share", "prior_variant_share"]
X, y, g = np.log1p(data[HIST].values.astype(float)), data.y.values, data.group.values

def folds(y, groups, k):
    k = int(min(k, y.sum(), (1 - y).sum()))
    return StratifiedGroupKFold(k, shuffle=True, random_state=SEED) if k >= 2 else None

def cv_scores(X, y, groups, make_model, k=5):
    """Out-of-fold scores from grouped, stratified cross-validation."""
    out = np.full(len(y), np.nan)
    for tr, te in folds(y, groups, k).split(X, y, groups):
        out[te] = make_model().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return out

def auc_ci(y, s, n=1000):
    rng = np.random.default_rng(SEED); vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():
            vals.append(roc_auc_score(y[i], s[i]))
    return roc_auc_score(y, s), np.percentile(vals, [2.5, 97.5])

ENOUGH = y.sum() >= 4 and folds(y, g, 5) is not None
print("groups:", len(set(g)), "| enough manipulated posts to fit models:", ENOUGH)

## 4. M0: detector score

Filled in once FLAME and TruFor scores exist in `results/m0_scores.csv` (columns: `img`, `score`).
The raw score is recalibrated on Reddit images with a one-variable logistic regression.

In [ ]:
m0_file = RESULTS / "m0_scores.csv"
if m0_file.exists() and ENOUGH:
    data = data.merge(pd.read_csv(m0_file), on="img", how="left").rename(columns={"score": "m0_raw"})
    data["m0"] = cv_scores(data[["m0_raw"]].values, y, g, lambda: LogisticRegression())
    auc, ci = auc_ci(y, data.m0.values)
    print(f"M0 AUROC {auc:.3f} (95% interval {ci[0]:.3f} to {ci[1]:.3f})")
else:
    print("No detector scores yet.")

## 5. M1: history models compared

Four model families, each tuned by grid search. The search runs *inside* each training part of the outer
grouped folds (nested cross-validation), so the score on the held-out part never helped choose the settings.

| Model | Settings searched | Class weighting |
|---|---|---|
| Logistic regression | `C` (regularization strength) | `class_weight="balanced"` |
| Decision tree | depth, smallest leaf | `class_weight="balanced"` |
| Random forest | depth, smallest leaf | `class_weight="balanced"` |
| XGBoost | learning rate, depth | `scale_pos_weight` = authentic / manipulated |

Logistic regression is the model the proposal fixes in advance. The others are extra comparisons, so picking
the best of them on these same folds is slightly optimistic.

In [ ]:
pos_weight = (len(y) - y.sum()) / max(y.sum(), 1)
MODELS = {
    "logistic regression": (make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000)),
                            {"logisticregression__C": [0.01, 0.1, 1, 10]}),
    "decision tree": (DecisionTreeClassifier(class_weight="balanced", random_state=SEED),
                      {"max_depth": [2, 3, 4], "min_samples_leaf": [1, 5, 10]}),
    "random forest": (RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=3),
                      {"max_depth": [2, 4, None], "min_samples_leaf": [1, 5]}),
    "xgboost": (XGBClassifier(n_estimators=200, scale_pos_weight=pos_weight, eval_metric="logloss",
                              random_state=SEED, n_jobs=3),
                {"learning_rate": [0.03, 0.1], "max_depth": [2, 3, 4]}),
}

def search(model, grid, X, y, g):
    return GridSearchCV(clone(model), grid, scoring="average_precision", cv=folds(y, g, 3), n_jobs=1)

def nested_cv(model, grid):
    """Outer grouped folds give the score; the grid search picks settings on the training part only."""
    scores, chosen = np.full(len(y), np.nan), []
    for tr, te in folds(y, g, 5).split(X, y, g):
        s = search(model, grid, X[tr], y[tr], g[tr]).fit(X[tr], y[tr], groups=g[tr])
        scores[te] = s.predict_proba(X[te])[:, 1]
        chosen.append(s.best_params_)
    return scores, chosen

rows = []
if ENOUGH:
    for name, (model, grid) in MODELS.items():
        s, chosen = nested_cv(model, grid)
        auc, ci = auc_ci(y, s)
        rows.append({"model": name, "AUROC": auc, "low": ci[0], "high": ci[1],
                     "avg precision": average_precision_score(y, s), "settings chosen per fold": chosen})
        data["m1_" + name.replace(" ", "_")] = s
    results = pd.DataFrame(rows).set_index("model")
    print(f"base rate (average precision of a random ranking): {y.mean():.3f}")
    display(results.drop(columns="settings chosen per fold").round(3))
else:
    print(f"Only {y.sum()} manipulated posts: too few to compare models yet.")

## 6. M1: the chosen history model

The best family from the table is tuned once more on all the data, and its feature weights are shown.
`m1` stays the logistic regression, as the proposal fixes; `m1_best` is the winner of the table.

In [ ]:
if ENOUGH:
    data["m1"] = data["m1_logistic_regression"]
    best = results["AUROC"].idxmax()
    data["m1_best"] = data["m1_" + best.replace(" ", "_")]
    final = search(*MODELS[best], X, y, g).fit(X, y, groups=g)
    est = final.best_estimator_
    weights = est[-1].coef_[0] if hasattr(est, "steps") else est.feature_importances_
    print(f"best: {best}, settings {final.best_params_}")
    display(pd.Series(weights, HIST, name="coefficient" if hasattr(est, "steps") else "importance").sort_values().round(3))

## 7. M2: image and history combined

**Confirmatory (late fusion).** `logit p = a * logit(p_M0) + b * logit(p_M1) + c`, fitted on the training folds.
History acts as a prior that shifts the detector's odds. The size of `b` is the question of the project.

**Exploratory (stacking).** XGBoost over the raw detector score and the history features together.
It is always reported next to the same model with the history columns shuffled.

In [ ]:
logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
if "m0" in data and "m1" in data:
    Z = np.c_[logit(data.m0), logit(data.m1)]
    data["m2"] = cv_scores(Z, y, g, lambda: LogisticRegression(class_weight="balanced"))
    auc, ci = auc_ci(y, data.m2.values)
    print(f"M2 late fusion AUROC {auc:.3f} (95% interval {ci[0]:.3f} to {ci[1]:.3f})")
    fit = LogisticRegression(class_weight="balanced").fit(Z, y)
    print("a (detector) = %.2f, b (history) = %.2f" % tuple(fit.coef_[0]))

    S = np.c_[data[["m0_raw"]].values, X]
    stack = lambda: XGBClassifier(n_estimators=200, max_depth=2, learning_rate=0.05,
                                  scale_pos_weight=pos_weight, eval_metric="logloss", random_state=SEED)
    data["m2_stack"] = cv_scores(S, y, g, stack)
    print(f"M2 stacking AUROC {roc_auc_score(y, data.m2_stack):.3f}")
else:
    print("Needs M0 (detector scores) and M1.")

## 8. Checks

Shuffled history must fall to chance: history columns are permuted within each community, which breaks the
link between a post and its poster's past while keeping each community's mix of values.

In [ ]:
if ENOUGH:
    rng = np.random.default_rng(1)
    Xs = X.copy()
    for s in data.subreddit.unique():
        idx = np.where(data.subreddit.values == s)[0]
        Xs[idx] = Xs[rng.permutation(idx)]
    lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000))
    print("shuffled-history AUROC %.3f" % roc_auc_score(y, cv_scores(Xs, y, g, lr)))

## 9. Save scores

In [ ]:
data.to_csv(RESULTS / "scores.csv", index=False)